### Model tuning

이전의 모델 학습, 튜닝 및 평가 과정은 `RandomizedSearch` 를 통하여 하이퍼 파라미터 튜닝을 진행
- 선정된 모델과 하이퍼 파라미터를 기반으로 해당 파라미터 공간 주변에서 `GridSearch`를 진행하여 튜닝을 한 번 더 진행
- 최종 모델 선정

In [6]:
import sys; sys.path.append('../src')
import numpy as np, joblib
from ssl_models import PARAM_SPACES
from ssl_train import tune_model, fit_estimator
from ssl_preprocessing import preprocess_product
from ssl_evaluate import infer_pos_label, get_scores, compute_metrics

#### **CN7**

In [ ]:
rf_grid_cn7 = {
    "n_estimators": [400],                    # 고정 (포화 구간)
    "max_depth": [None],                      # 고정
    "min_samples_leaf": [1, 2, 3],            # 1이 경계였으므로 이웃 값 확인
    "max_features": ["sqrt", 0.15, 0.3],
    "criterion": ["gini", "entropy"],         # 저렴한 추가 축
}

In [ ]:
PARAM_SPACES["rf"] = rf_grid_cn7
out = preprocess_product("cn7", "processed2", scaling=None, feature_selection=None,
                         imbalance="class_weight", verbose=False)       # 또는 "class_weight"
X = out.X_train_orig.to_numpy(float); y = np.asarray(out.y_train_orig)
X_u = out.X_unlabeled.to_numpy(float); pos = infer_pos_label(y)
kw = dict(imbalance=out.imbalance, smote_params=out.smote_params, semi_supervised="self_training")

best_params, best_cv, cv_df = tune_model("rf", X, y, X_u, pos_label=pos, scoring="f1",
                                         cv=5, search="grid", n_jobs=3, **kw)
est = fit_estimator("rf", best_params, X, y, X_u, **kw)

X_te = out.X_test.to_numpy(float)
print(compute_metrics(np.asarray(out.y_test), est.predict(X_te), get_scores(est, X_te, pos), pos))
joblib.dump({"model": est, "preprocessor": out.preprocessor, "pos_label": pos,
             "best_params": best_params}, "../models/rf_cn7_best.joblib")

#### **RG3**

In [ ]:
# randomized search 결과 기준 파라미터 탐색 공간 세팅
rf_grid_rg3 = {
    "n_estimators": [500],                    # 고정 (포화 구간)
    "max_depth": [None],                      # 고정
    "min_samples_leaf": [1, 2, 3],            # 1이 경계였으므로 이웃 값 확인
    "max_features": ["log2", 0.1, 0.2, 0.3],
    "criterion": ["gini", "entropy"],         # 저렴한 추가 축
}


===== [1/1] rg3/processed2 preprocessing: {'scaling': None, 'feature_selection': None, 'imbalance': 'class_weight'} =====


KeyboardInterrupt: 

In [ ]:
PARAM_SPACES["rf"] = rf_grid_rg3
out = preprocess_product("rg3", "processed2", scaling=None, feature_selection=None,
                         imbalance="class_weight", verbose=False)       # 또는 "class_weight"
X = out.X_train_orig.to_numpy(float); y = np.asarray(out.y_train_orig)
X_u = out.X_unlabeled.to_numpy(float); pos = infer_pos_label(y)
kw = dict(imbalance=out.imbalance, smote_params=out.smote_params, semi_supervised="self_training")

best_params, best_cv, cv_df = tune_model("rf", X, y, X_u, pos_label=pos, scoring="f1",
                                         cv=5, search="grid", n_jobs=3, **kw)
est = fit_estimator("rf", best_params, X, y, X_u, **kw)

X_te = out.X_test.to_numpy(float)
print(compute_metrics(np.asarray(out.y_test), est.predict(X_te), get_scores(est, X_te, pos), pos))
joblib.dump({"model": est, "preprocessor": out.preprocessor, "pos_label": pos,
             "best_params": best_params}, "../models/rf_rg3_best.joblib")